In [2]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import sys
from pathlib import Path
import importlib

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import os
import logging
from datetime import datetime

import data.schema_mapping as schema_mapping
schema_mapping = importlib.reload(schema_mapping)
KAGGLE_TO_STANDARD = schema_mapping.KAGGLE_TO_STANDARD
GBP_TO_PKR = schema_mapping.GBP_TO_PKR
FILTERS = schema_mapping.FILTERS
OUTPUT_COLUMNS = schema_mapping.OUTPUT_COLUMNS

In [6]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

log = logging.getLogger(__name__)

In [11]:
BASE_DIR = Path.cwd().parent
RAW_PATH = BASE_DIR / "data" / "raw" / "online_retail_II.csv"
PROCESSED_PATH = BASE_DIR / "data" / "processed" / "retail_cleaned.csv"

In [17]:
def load_raw_data(path=RAW_PATH):
    log.info(f"Loading raw data from: {path}")

    df = pd.read_csv(path, dtype={"Customer ID": str})

    log.info(f"Loaded {len(df):,} rows")
    return df


df = load_raw_data()
df.head()

2026-06-13 20:51:57,137 | INFO | Loading raw data from: c:\Langchain\ai-retail-intelligence\data\raw\online_retail_II.csv
2026-06-13 20:51:59,025 | INFO | Loaded 1,067,371 rows


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [20]:
def rename_columns(df):
    df = df.rename(columns=KAGGLE_TO_STANDARD)
    log.info("Columns renamed")
    return df


df = rename_columns(df)
df.head()

2026-06-13 20:52:45,006 | INFO | Columns renamed


,invoice_id,product_code,product_name,quantity,sale_date,unit_price_gbp,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [21]:
def clean_data(df):
    original_len = len(df)

    df = df.dropna(subset=["product_name", "unit_price_gbp"])

    df = df[df["quantity"] >= FILTERS["min_quantity"]]
    df = df[df["unit_price_gbp"] >= FILTERS["min_unit_price"]]

    df = df[df["product_code"].str.match(r"^\d", na=False)]

    df["product_name"] = df["product_name"].str.strip().str.title()

    df["sale_date"] = pd.to_datetime(df["sale_date"]).dt.date

    log.info(f"Cleaned: {original_len} → {len(df)} rows")

    return df


df = clean_data(df)
df.head()

2026-06-13 20:53:04,666 | INFO | Cleaned: 1067371 → 1036877 rows


,invoice_id,product_code,product_name,quantity,sale_date,unit_price_gbp,customer_id,country
0,489434,85048,15Cm Christmas Glass Ball 20 Lights,12,2009-12-01,6.95,13085.0,United Kingdom
1,489434,79323P,Pink Cherry Lights,12,2009-12-01,6.75,13085.0,United Kingdom
2,489434,79323W,White Cherry Lights,12,2009-12-01,6.75,13085.0,United Kingdom
3,489434,22041,"Record Frame 7"" Single Size",48,2009-12-01,2.10,13085.0,United Kingdom
4,489434,21232,Strawberry Ceramic Trinket Box,24,2009-12-01,1.25,13085.0,United Kingdom


In [22]:
def categorize_product(name):
    name = name.lower()

    rules = {
        "Tea & Beverages": ["tea", "coffee", "juice", "drink", "water", "cup"],
        "Snacks & Food": ["cake", "biscuit", "chocolate", "sweet", "snack"],
        "Home & Kitchen": ["glass", "bowl", "plate", "kitchen", "jar"],
        "Clothing": ["bag", "shirt", "dress"],
        "Decoration": ["candle", "light", "frame", "flower"],
        "Stationery": ["pen", "book", "card"],
        "Personal Care": ["soap", "cream", "lotion"],
    }

    for category, keywords in rules.items():
        if any(k in name for k in keywords):
            return category

    return "General"


def transform_data(df):
    df["unit_price_pkr"] = (df["unit_price_gbp"] * GBP_TO_PKR).round(2)
    df["category"] = df["product_name"].apply(categorize_product)

    log.info("Transformation complete")
    return df


df = transform_data(df)
df.head()

2026-06-13 20:56:28,502 | INFO | Transformation complete


,invoice_id,product_code,product_name,quantity,sale_date,unit_price_gbp,customer_id,country,unit_price_pkr,category
0,489434,85048,15Cm Christmas Glass Ball 20 Lights,12,2009-12-01,6.95,13085.0,United Kingdom,2432.5,Home & Kitchen
1,489434,79323P,Pink Cherry Lights,12,2009-12-01,6.75,13085.0,United Kingdom,2362.5,Decoration
2,489434,79323W,White Cherry Lights,12,2009-12-01,6.75,13085.0,United Kingdom,2362.5,Decoration
3,489434,22041,"Record Frame 7"" Single Size",48,2009-12-01,2.10,13085.0,United Kingdom,735.0,Decoration
4,489434,21232,Strawberry Ceramic Trinket Box,24,2009-12-01,1.25,13085.0,United Kingdom,437.5,General


In [23]:
def select_output_columns(df):
    cols = [c for c in OUTPUT_COLUMNS if c in df.columns]
    return df[cols]


df = select_output_columns(df)
df.head()

,product_code,product_name,category,quantity,unit_price_pkr,sale_date,invoice_id
0,85048,15Cm Christmas Glass Ball 20 Lights,Home & Kitchen,12,2432.5,2009-12-01,489434
1,79323P,Pink Cherry Lights,Decoration,12,2362.5,2009-12-01,489434
2,79323W,White Cherry Lights,Decoration,12,2362.5,2009-12-01,489434
3,22041,"Record Frame 7"" Single Size",Decoration,48,735.0,2009-12-01,489434
4,21232,Strawberry Ceramic Trinket Box,General,24,437.5,2009-12-01,489434


In [24]:
def save_processed(df):
    os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)
    df.to_csv(PROCESSED_PATH, index=False)
    log.info(f"Saved → {PROCESSED_PATH}")


save_processed(df)

2026-06-13 20:56:55,842 | INFO | Saved → c:\Langchain\ai-retail-intelligence\data\processed\retail_cleaned.csv


In [29]:
def load_into_database(df):
    import importlib
    import database.db_manager as db_manager
    db_manager = importlib.reload(db_manager)
    get_connection = db_manager.get_connection
    db_manager.initialize_database()

    conn = get_connection()
    cursor = conn.cursor()

    inserted_products = 0
    inserted_sales = 0

    for _, row in df.iterrows():

        cursor.execute(
            "SELECT product_id FROM products WHERE product_name = ?",
            (row["product_name"],)
        )
        result = cursor.fetchone()

        if result is None:
            cursor.execute("""
                INSERT INTO products
                (product_name, category, cost_price, selling_price, stock)
                VALUES (?, ?, ?, ?, ?)
            """, (
                row["product_name"],
                row["category"],
                round(row["unit_price_pkr"] * 0.75, 2),
                row["unit_price_pkr"],
                0
            ))
            product_id = cursor.lastrowid
            inserted_products += 1
        else:
            product_id = result[0]

        cursor.execute("""
            INSERT INTO sales (product_id, quantity, sale_date)
            VALUES (?, ?, ?)
        """, (
            product_id,
            int(row["quantity"]),
            str(row["sale_date"])
        ))

        inserted_sales += 1

    conn.commit()
    conn.close()

    print(f"Products inserted: {inserted_products}")
    print(f"Sales inserted: {inserted_sales}")


load_into_database(df.head(100))  # start small for notebook

✅ Database initialized successfully.
Products inserted: 90
Sales inserted: 100


In [ ]:
def run_pipeline():
    df = load_raw_data()
    df = rename_columns(df)
    df = clean_data(df)
    df = transform_data(df)
    df = select_output_columns(df)
    save_processed(df)

    load_into_database(df.head(500))

    return df


final_df = run_pipeline()
final_df.head()